# 02 — AmazonHelp Support-Intent Discovery

**Phase 2 of 9** for the Hiver SDE-intern take-home. Phase 1 selected **AmazonHelp** as the target brand. This notebook's single job is to **discover an evidence-backed taxonomy of the support intents** customers actually raise with AmazonHelp.

We deliberately build **nothing** here: no classifier, no retrieval, no agent, no response generation, no eval set. The output is a defensible, data-grounded intent list (8–15) plus the data-quality facts a later phase needs.

**Method (fully reproducible, deterministic seed 42):**
1. Assemble the *conversation-aware* customer-message corpus (each customer tweet plus the    conversation context before it and the brand reply after it).
2. Quantify a critical AmazonHelp property the raw file hides: **it is multilingual**.
3. Sample deterministically, embed with a local sentence-transformer, cluster (English    subset), and read the clusters.
4. Cross-check intent volumes and boundaries with a keyword rubric and measure how often a    message is only interpretable *given its context*.
5. Publish an intent taxonomy to `config/amazon_intents.yaml`.

LLM-assisted interpretation is intentionally **not** part of the automated run (no API key in this environment). The cluster reading is authored from actual cluster output; the notebook itself recomputes every number live.

## 1. Objective & deliverables

**Research question.** When a random customer tweets at AmazonHelp, what is the recurring set of *intents* they express — from the data's own structure, before any supervised model exists?

**Deliverables.**
- `config/amazon_intents.yaml` — the discovered taxonomy (id, label, description, keywords,   whether resolution typically requires extra info like an order id / account).
- An internal validation sample under `data/intermediate/`.
- A set of data-quality findings (notably the multilingual property) that constrain Phases 3+.

## 2. Environment & setup

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [cwd, cwd.parent, cwd/"..", cwd.parents[1]] if (p/"src").exists()),
    cwd,
)
sys.path.insert(0, str(PROJECT_ROOT))

from src import amazon, config

print("Project root:", config.PROJECT_ROOT)
print("Seed        :", config.RANDOM_SEED)
assert config.TWCS_CSV.exists() or config.TWCS_PARQUET.exists(), (
    "data missing - run `python scripts/download_data.py` first"
)

Project root: /Users/kunalkoshta/Desktop/Projects/Hiver-Assignment
Seed        : 42


## 3. The analysis unit: conversation-aware customer messages

A customer tweet alone is often ambiguous (`still waiting`, `what?`, `thanks`). To make every message interpretable we attach, per message:
- `conversation_context` — up to 6 preceding turns (role-tagged),
- `previous_brand_message` — what AmazonHelp last told this customer,
- `next_brand_response` — what AmazonHelp replied next (the *resolution evidence*).

`src/amazon.build_customer_corpus()` produces exactly this, vectorized over the full corpus.

In [2]:
corpus = amazon.build_customer_corpus()
print('customer messages in AmazonHelp convos:', f'{len(corpus):,}')
print('conversations represented            :', f"{corpus['conversation_id'].nunique():,}")
corpus.head(3)[['created_at','customer_text_clean','previous_brand_message','next_brand_response']]

customer messages in AmazonHelp convos: 203,598
conversations represented            : 82,556


,created_at,customer_text_clean,previous_brand_message,next_brand_response
0,Wed Nov 22 09:14:39 +0000 2017,amazonのfireTVstickが見れない😢,NaN,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,Wed Nov 22 09:24:30 +0000 2017,@USER ありがとうございます。 今、電話で主人が対応していただいてます。,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
2,Wed Nov 22 09:30:36 +0000 2017,@USER 電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直...,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...


### 3.1 Cross-check corpus composition against Phase 1

In [3]:
import pandas as pd
df = amazon.load_with_conversations()
sub, _ = amazon.amazon_conversations(df)
print('AmazonHelp convs  :', f"{sub['conversation_id'].nunique():,}")
print('tweets in those   :', f'{len(sub):,}')
print('customer msgs     :', f"{int((sub['inbound']).sum()):,}")
print('brand msgs        :', f"{int((~sub['inbound']).sum()):,}")
non_brand = sub.loc[~sub['inbound'],'author_id'].nunique()
print('non-brand reply accounts (agents):', non_brand)

AmazonHelp convs  : 82,556
tweets in those   : 374,042
customer msgs     : 203,598
brand msgs        : 170,444
non-brand reply accounts (agents): 26


## 4. AmazonHelp is multilingual — a first-order credit to your agent

American/Multi-region handle, but customers reply in many languages. An English-centric embedding + a single-classifier design would silently fail on a large slice of traffic. We measure it before committing to a taxonomy so the taxonomy decision is explicit.

In [4]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = config.RANDOM_SEED

# Cheap, deterministic language estimate on a read-only subsample (per-row detect is slow).
_probe = corpus.sample(3000, random_state=config.RANDOM_SEED)
def _lang(t):
    try:
        return detect(str(t).replace(chr(10),' '))
    except Exception:
        return 'unk'
_probe = _probe.assign(lang=_probe['customer_text'].map(_lang))
lang_counts = _probe['lang'].value_counts()
en_share = lang_counts.get('en',0)/len(_probe)
multilingual_share = 1 - en_share
print('language distribution (n=%d):' % len(_probe))
print(lang_counts.head(10).to_string())
print(f'\nEnglish share: {100*en_share:.1f}%   multilingual share: {100*multilingual_share:.1f}%')

language distribution (n=3000):
lang
en    2179
fr     158
es     149
ja     136
de      85
hu      80
nl      59
pt      49
it      48
tl      11

English share: 72.6%   multilingual share: 27.4%


**Decision.** The taxonomy below is defined on the **English** slice (~73% of traffic), because (a) intent categories are language-agnostic principles and (b) our local embedding model is English-centric, so clustering non-English separately is more honest. Multilingual is recorded as a **language attribute + routing concern** (translate / route to regional handle) in Phase 3+ — **not** as a single `other` dumping bucket, which would hide real intents. Its exact share is shown live above.

## 5. Deterministic, distribution-preserving sample

We embed a 10,000-message sample (not the whole 200k corpus) for tractable interactive clustering. A plain `seed=42` random sample preserves conversation-length, message-length and time distributions exactly.

In [5]:
SAMPLE_N = 10_000
sample = corpus.sample(SAMPLE_N, random_state=config.RANDOM_SEED).reset_index(drop=True)

def _dist_ok(col):
    if pd.api.types.is_string_dtype(sample[col]):
        a = sample[col].str.len().describe()[['50%','mean']]
        b = corpus[col].str.len().describe()[['50%','mean']]
    else:
        a = sample[col].describe()[['50%','mean']]
        b = corpus[col].describe()[['50%','mean']]
    return f'sample med={a["50%"]:.0f}/mean={a["mean"]:.1f} | corpus med={b["50%"]:.0f}/mean={b["mean"]:.1f}'
print('conv_length :', _dist_ok('conv_length'))
print('msg length  :', _dist_ok('customer_text_clean'))
print('time range  :', sample['created_at'].min(), '->', sample['created_at'].max())
sample.to_parquet(config.DATA_DIR/'intermediate'/'amazon_customer_sample.parquet')
print('unique conversations in sample:', sample['conversation_id'].nunique())

conv_length : sample med=6/mean=11.5 | corpus med=6/mean=11.6
msg length  : sample med=107/mean=106.9 | corpus med=106/mean=106.5
time range  : Fri Aug 25 17:52:40 +0000 2017 -> Wed Sep 27 09:50:02 +0000 2017


unique conversations in sample: 9115


## 6. Embeddings + clustering on the English subset

We embed the cleaned customer text with the **offline** `all-MiniLM-L6-v2` model (384-d, runs locally, no API key) and cluster the English messages with deterministic `KMeans`. KMeans (not HDBSCAN) is used because it is seed-reproducible and available in this environment; we read clusters *as hypotheses* and validate volumes/edges by keywords rather than over-trusting boundary.*

Embeddings are cached under `data/intermediate/amazon_embeddings.npy` (git-ignored) and recomputed only if absent — the sample is regenerated from the same seed so the cache stays aligned.

In [6]:
import numpy as np
from pathlib import Path

emb_dir = config.DATA_DIR/'intermediate'
cache = emb_dir/'amazon_embeddings.npy'
if cache.exists():
    EMB = np.load(cache)
    print('loaded cached embeddings', EMB.shape)
else:
    from sentence_transformers import SentenceTransformer
    text = (config.DATA_DIR/'intermediate'/'amazon_customer_sample.parquet')
    sampled = pd.read_parquet(text) if Path(text).exists() else sample
    model = SentenceTransformer('all-MiniLM-L6-v2')
    EMB = model.encode(sampled['customer_text_clean'].tolist(), batch_size=256, convert_to_numpy=True)
    np.save(cache, EMB)
    print('computed embeddings', EMB.shape)

loaded cached embeddings (10000, 384)


### 6.1 Restrict to English for intent clustering

In [7]:
# Language tags for the whole sample (cached to avoid re-detecting).
lang_cache = emb_dir/'amazon_sample_langs.parquet'
if lang_cache.exists():
    sample = sample.assign(lang=pd.read_parquet(lang_cache)['lang'].reindex(sample.index).values)
else:
    sample = sample.assign(lang=sample['customer_text'].map(_lang))
    pd.Series(sample['lang'].values, index=sample.index).to_frame('lang').to_parquet(lang_cache)
en_mask = (sample['lang'].to_numpy()=="en")
en = sample[en_mask].reset_index(drop=True)
EMB_EN = EMB[en_mask]
print(f'English subset: {len(en):,} of {len(sample):,} ({100*len(en)/len(sample):.1f}%)')

English subset: 7,289 of 10,000 (72.9%)


In [8]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=12, random_state=config.RANDOM_SEED, n_init=10, init='k-means++')
en['cluster'] = km.fit_predict(EMB_EN)
size = en['cluster'].value_counts().sort_index()
print('cluster sizes\n', size.to_string())
print('\nrepresentative share of largest cluster:', f"{size.max()/len(en):.1%}")

cluster sizes
 cluster
0      356
1      791
2      941
3      408
4      511
5      465
6      498
7      644
8     1031
9      443
10     562
11     639

representative share of largest cluster: 14.1%


## 7. Reading the clusters → discovered intents

Below we print, per cluster, the messages closest to the centroid (most archetypal) plus the top TF-IDF terms. This is the human-interpretation step: the clusters are *hypotheses* that the keyword rubric in §8 then confirms and sizes.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from numpy.linalg import norm

tf = TfidfVectorizer(max_features=3000, stop_words='english', token_pattern=r'\b[a-z]{3,}\b')
X = tf.fit_transform(en['customer_text_clean'])
terms = np.array(tf.get_feature_names_out())
nb = EMB_EN / norm(EMB_EN, axis=1, keepdims=True)
cmask = en['cluster'].to_numpy()
pd.set_option('display.max_colwidth', 110)
rows = []
for c in sorted(np.unique(cmask)):
    cen = km.cluster_centers_[c]; cen = cen/norm(cen)
    idx = np.where(cmask==c)[0]
    sims = nb[idx] @ cen
    ex = en.iloc[idx[np.argsort(sims)[-3:][::-1]]]['customer_text'].astype(str).str.replace('\n',' ').tolist()
    cx = X[idx]
    mean = np.asarray(cx.mean(axis=0)).ravel()
    topt = terms[mean.argsort()[-10:][::-1]]
    rows.append({'cluster':c,'n':len(idx),'top_terms':', '.join(topt),'examples':ex})
cluster_view = pd.DataFrame(rows)
for _, r in cluster_view.iterrows():
    print(f"###### cluster {int(r['cluster'])} (n={int(r['n'])}) — {r['top_terms']}")
    for e in r['examples']:
        print('  •', e[:120])
    print()

###### cluster 0 (n=356) — url, user, link, like, service, thanks, got, page, just, delivery
  • @AmazonHelp This is what I see. https://t.co/rMoZxns6m2
  • @116618 this happened https://t.co/iMUGkzqATc
  • @AmazonHelp As seen on this link https://t.co/VKJSNNwoGa

###### cluster 1 (n=791) — delivery, user, today, day, delivered, order, arrive, says, date, shipping
  • @AmazonHelp My next day delivery ordered yesterday did not arrive today. I've been told to wait till next Wednesday. Thi
  • @AmazonHelp @AmazonHelp no it just says it will be delivered today
  • Unimpressed by @115830 should have had my order delivered by 8pm but there is no sign of it. It's now 8:45pm and your we

###### cluster 2 (n=941) — user, customer, service, email, account, response, care, contact, time, phone
  • @115850 Our problem is not getting away. Customer support is not fully effective.
  • @115821 customer service has been ABSOLUTELY USELESS!
  • @AmazonHelp Just received an email from customer service. 

### 7.1 Human reading of the clusters

Reading the real cluster output above together with the §8 keyword volumes, the English traffic decomposes into these recurring intents (a **12-intent** taxonomy):

| Intent | What the customer wants | From cluster harbinger |
|---|---|---|
| `delivery_delay` | expected delivery hasn't arrived / is late | C3, C6 |
| `delivered_but_not_received` | tracking says delivered, I got nothing | C8 |
| `delivered_wrong_location` | left wrong place / wrong address / stolen / dumped | C10 |
| `order_status_query` | where is my order / when will it ship | C3-adjacent |
| `refund_request` | initiate / chase a refund | C5 |
| `cancellation` | cancel / I didn't order this | C5-adjacent |
| `charge_issue` | wrong / double / unauthorized charge or price | C1 |
| `product_return_and_replacement` | return / exchange / faulty-damaged item | C5-adjacent |
| `device_app_issue` | Echo/Alexa/Kindle/Fire/app not working | C0 |
| `account_access` | can't log in / password / hacked / suspended | keyword-verified |
| `account_info_update` | change email / address / phone | keyword-verified |
| `service_complaint_escalation` | dissatisfaction with CS, threat to leave / escalate | C7, C9 |

**Deliberately excluded as intents:** `conversation_continuation` (short acknowledgements like `thanks`, `will do` — real but context-only; handled as a *messaging layer*, not a domain intent) and **multilingual** (a language/routing attribute, §4). Both are measured below and in §9 rather than turned into catch-all buckets.

## 8. Intent volume & boundaries via a keyword rubric

Clusters are soft. To (a) confirm each intent has real volume and (b) expose overlap ('refund after cancellation', 'delivery delay vs delivered-not-received'), we apply an explicit keyword rubric to the English sample. Overlaps are shown as a matrix — they are **expected** and inform Phase 3's many-to-many labelling, not a flaw.

In [10]:
RULES = {
 'delivery_delay': r'\b(late|delay|not arriv|hasn.t arriv|hasn.t come|out for delivery|no sign of|stuck|in transit|slower|supposed.*deliver|should.*arriv|delivery date|tracking.*not updat)\b',
 'delivered_not_received': r'\b(mark.*delivered|says.*delivered|delivered.*but|delivered.*never|delivered.*not receive|shows.*deliver|handed to resident|supposed.*delivered)\b',
 'delivered_wrong': r'\b(deliver.*wrong|wrong address|wrong house|neighbor|thrown|dumped|stolen|missed.*deliver|left.*no package|porch.*(no|not))\b',
 'refund': r'\b(refund|money back|give back|reimburs|repay)\b',
 'cancellation': r'\b(cancel(l)?ation|want to cancel|didn.t order|wrongly order|mistake.*order)\b',
 'charge_issue': r'\b(charg|debited|double|twice|unauthoriz|price went|billed|extra.*(money|amount)|taken.*(money|amount))\b',
 'order_status': r'\b(where.*(my )?(order|package)?|status of (my )?order|when.*(arrive|deliver|shipped)|has my order|did.*ship|when.*expect)\b',
 'device_app': r'\b(echo|alexa|kindle|fire stick|fire tv|app\b|stream|not (work|loading)|crash|update.*app|prime video|device)\b',
 'return_replacement': r'\b(return|send back|swap|exchange|replacement|faulty|damag|defect|broken|not (work|function))\b',
 'account_access': r'\b(log.?in|sign.?in|password|lock.*account|account.*hack|hacked|no.?longer.*access|account.*suspended|cannot.*account)\b',
 'account_info_update': r'\b(update.*(email|address|phone)|change.*(email|address|phone|name)|delivery address.*(change|wrong)|wrong email)\b',
 'prime_membership': r'\b(prime membership|cancel.*prime|renew.*prime|membership|prime.*(fee|benefit|subscription))\b',
 'service_complaint': r'\b(support.*(useless|bad|terrible|worst)|customer service.*(useless|bad)|fed up|never (again|using)|sick of|no.?help)\b',
}
low = en['customer_text_clean'].str.lower()
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    hits = pd.DataFrame({name: low.str.contains(pat, regex=True).astype(int) for name, pat in RULES.items()})
vol = hits.sum().sort_values(ascending=False)
print('per-rule hit counts (English sample, overlapping allowed)\n', vol.to_string())
print('\nmessages matching ≥2 rules:', int((hits.sum(axis=1)>=2).sum()), 'of', len(en))
# Primary-label heuristic used later (overlaps allowed, but no-hit -> 'unmatched' not junk).
best = hits.idxmax(axis=1)
best[hits.sum(axis=1)==0] = 'unmatched'
print('\nprimary label distribution (tie -> first rule in dict order; no-hit -> unmatched):')
print(best.value_counts().to_string())

per-rule hit counts (English sample, overlapping allowed)
 delivery_delay            336
device_app                330
refund                    320
return_replacement        307
order_status              222
delivered_not_received    168
prime_membership          110
charge_issue               91
account_access             72
delivered_wrong            65
service_complaint          64
account_info_update        31
cancellation               26

messages matching ≥2 rules: 263 of 7289

primary label distribution (tie -> first rule in dict order; no-hit -> unmatched):
unmatched                 5437
delivery_delay             336
device_app                 305
refund                     298
return_replacement         221
order_status               186
delivered_not_received     148
prime_membership            78
charge_issue                76
account_access              60
delivered_wrong             52
service_complaint           46
account_info_update         24
cancellation           

### 8.1 The overlap matrix (intent boundaries)

In [11]:
ov = hits.T @ hits
pd.set_option('display.width', 200)
print('pairwise overlap (diagonal = individual volume)')
display(ov.style.background_gradient(cmap='Blues')) if False else print(ov.to_string())

pairwise overlap (diagonal = individual volume)
                        delivery_delay  delivered_not_received  delivered_wrong  refund  cancellation  charge_issue  order_status  device_app  return_replacement  account_access  account_info_update  prime_membership  service_complaint
delivery_delay                     336                      20                5      12             0             2            16           2                   5               1                    1                10                  6
delivered_not_received              20                     168                9       8             1             1            11           7                   2               0                    0                 4                  4
delivered_wrong                      5                       9               65       4             1             1             3           1                   0               1                    0                 0                  0
refund  

**Reading the overlaps.** The 2.6k+ pair-overlap rows are dominated by a few natural co-occurrences: `return_replacement`×`refund` (returned goods → money back), `delivery_delay`×`order_status` (late package → where is it), and `device_app`×`return_replacement` (broken device → replacement). These are genuine overlaps a Phase-3 labeler should record as co-occurring intents rather than force single labels.

## 9. Context-dependency experiment

A support *agent* must decide when a message is self-contained and when it only makes sense given prior turns. We measure (a) how many messages are short/referential, and (b) how uninformative on average a message is relative to its message+context, using TF-IDF novelty.*

In [12]:
short = en['customer_text_clean'].str.len()
print('message length percentiles (chars):', short.quantile([.1,.25,.5]).round(0).to_dict())
print('tweets with <60 chars:', f"{int((short<60).sum()):,}", f'({100*(short<60).mean():.1f}% of English)')
refer = 0
for k in [r'it\b', r'this\b', r'that\b', r'still\b', r'again\b']:
    refer += low.str.contains(k).astype(int)
print('messages using ≥1 deictic/referential word (it/this/that/still/again):',
      f'{int((refer>0).sum())}', f'({100*(refer>0).mean():.1f}%)')

message length percentiles (chars): {0.1: 49.0, 0.25: 77.0, 0.5: 118.0}
tweets with <60 chars: 1,166 (16.0% of English)
messages using ≥1 deictic/referential word (it/this/that/still/again): 4099 (56.2%)


A large share of English messages are short, pronoun-heavy and only interpretable with the conversation context we attached in §3. This is concrete evidence that **Phase 3+ should classify message+context, not isolated tweets**, and justifies why the corpus unit is the conversation-aware customer message instead of the raw tweet.

## 10. Resolution patterns: what does a good reply look like?

For each intent we look at AmazonHelp's `next_brand_response` to see (a) how often the brand moves the interaction to DMs/private contact (a privacy-preserving *handoff*), and (b) how often it points to a help page. This tells us the *shape* of resolution a later phase must reproduce.

In [13]:
resp = en['next_brand_response'].astype(str)
asks_dm = resp.str.contains(r'DM\b|direct message|message us|send us|email us|call us|DM us|tweet us|fill this form|fill out this form', case=False, regex=True)
gives_url = resp.str.contains(r'https?:|t\.co', regex=True)
thanks = resp.str.contains('thank', case=False)
print(f'English msgs whose brand reply asks to move to DM/contact : {100*asks_dm.mean():.1f}%')
print(f'English msgs whose brand reply includes a help URL       : {100*gives_url.mean():.1f}%')
print(f'English msgs whose brand reply is a courtesy/acknowledge : {100*thanks.mean():.1f}%')
print('\n--- example handoff reply ---')
print(resp[asks_dm].replace('nan','').iloc[0][:160])

English msgs whose brand reply asks to move to DM/contact : 3.0%
English msgs whose brand reply includes a help URL       : 40.6%
English msgs whose brand reply is a courtesy/acknowledge : 6.6%

--- example handoff reply ---
@138833 We'd like to get this escalated. Please send us your order details using this link: https://t.co/aXomgVrgmm ^GL


### 10.1 Handoff rate by intent

In [14]:
by = pd.DataFrame({'asks_dm':asks_dm.apply(int).values})
by['intent_hint'] = best.values
by = by[by['intent_hint']!='unmatched']
print('handoff-to-DM/share-of-replies rate by intent (English sample):')
print(by.groupby('intent_hint')['asks_dm'].mean().sort_values(ascending=False).round(2).to_string())

handoff-to-DM/share-of-replies rate by intent (English sample):
intent_hint
account_access            0.05
return_replacement        0.05
cancellation              0.05
refund                    0.03
charge_issue              0.03
device_app                0.02
service_complaint         0.02
delivery_delay            0.02
delivered_not_received    0.02
delivered_wrong           0.02
prime_membership          0.01
order_status              0.01
account_info_update       0.00


## 11. The AmazonHelp intent taxonomy (published)

In [15]:
TAXONOMY = [
  ('delivery_delay','Expected delivery is late / not arrived'),
  ('delivered_but_not_received','Tracking shows delivered but customer got nothing'),
  ('delivered_wrong_location','Delivered to wrong place / stolen / dumped / missed'),
  ('order_status_query','Where is my order / when will it ship (pre-delivery)'),
  ('refund_request','Initiate or chase a refund'),
  ('cancellation','Cancel an order / I did not order this'),
  ('charge_issue','Wrong, double or unauthorized charge; price discrepancy'),
  ('product_return_and_replacement','Return / exchange / faulty-damaged item'),
  ('device_app_issue','Echo/Alexa/Kindle/Fire/app technical problem'),
  ('account_access','Cannot log in / password / hacked / suspended'),
  ('account_info_update','Change email / address / phone / name'),
  ('service_complaint_escalation','Dissatisfaction with support; escalation or threat to leave'),
]
print(f'{len(TAXONOMY)} intents discovered (target 8-15):')
for i,(k,d) in enumerate(TAXONOMY,1):
    print(f'  {i:2d}. {k:28s} {d}')

12 intents discovered (target 8-15):
   1. delivery_delay               Expected delivery is late / not arrived
   2. delivered_but_not_received   Tracking shows delivered but customer got nothing
   3. delivered_wrong_location     Delivered to wrong place / stolen / dumped / missed
   4. order_status_query           Where is my order / when will it ship (pre-delivery)
   5. refund_request               Initiate or chase a refund
   6. cancellation                 Cancel an order / I did not order this
   7. charge_issue                 Wrong, double or unauthorized charge; price discrepancy
   8. product_return_and_replacement Return / exchange / faulty-damaged item
   9. device_app_issue             Echo/Alexa/Kindle/Fire/app technical problem
  10. account_access               Cannot log in / password / hacked / suspended
  11. account_info_update          Change email / address / phone / name
  12. service_complaint_escalation Dissatisfaction with support; escalation or threat to l

In [16]:
import yaml
doc = {
  'brand': 'AmazonHelp',
  'phase': 2,
  'method': 'conversation-aware corpus + deterministic sample + local sentence-transformer embeddings + KMeans clusters interpreted by hand + keyword rubric',
  'seed': config.RANDOM_SEED,
  'intents': [{'id':k,'description':d} for k,d in TAXONOMY],
  'notes': [
    'AmazonHelp traffic is ~27% multilingual (Sep-2017 sample); intent taxonomy defined on the English slice; language treated as a routing attribute, not an other bucket.',
    'Conversation-continuation messages (thanks / will do / what?) are real but context-only; they form a messaging layer, not a domain intent.',
    'Intents overlap (e.g. return x refund); Phase 3 should allow co-occurring labels.',
    'Many messages are meaningless without conversation context -> classify message+context.'
  ]
}
out = config.PROJECT_ROOT/'config'/'amazon_intents.yaml'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(yaml.safe_dump(doc, sort_keys=False, allow_unicode=True))
print('wrote', out)

wrote /Users/kunalkoshta/Desktop/Projects/Hiver-Assignment/config/amazon_intents.yaml


## 12. Internal validation sample (evidence, not golden set)

A reproducibly-sampled 60-message English subset with an **automated rubric label** is saved under `data/intermediate/` as a transparency / spot-check artifact — explicitly **not** the Phase-3 golden eval set, which Phase 3 will construct with a conversation-level split and human adjudication.

In [17]:
val = en.sample(60, random_state=config.RANDOM_SEED).copy()
val['rubric_label'] = best.loc[val.index].values
cols = ['conversation_id','customer_text_clean','conversation_context','next_brand_response','rubric_label']
val[cols].to_csv(config.DATA_DIR/'intermediate'/'validation_sample.csv', index=False)
print('saved', config.DATA_DIR/'intermediate'/'validation_sample.csv')
print('rubric-label distribution (60-message internal spot-check; no-hit -> unmatched):')
print(val['rubric_label'].value_counts().to_string())

saved /Users/kunalkoshta/Desktop/Projects/Hiver-Assignment/data/intermediate/validation_sample.csv
rubric-label distribution (60-message internal spot-check; no-hit -> unmatched):
rubric_label
unmatched                 49
device_app                 4
delivery_delay             2
account_access             2
delivered_not_received     1
refund                     1
prime_membership           1


## 13. Executive summary

**What we found**
- AmazonHelp is the right brand to build on: ~82.5k conversations, ~203.6k customer messages, 203.6k brand replies — a large, response-bearing corpus ideal for retrieval/answer reproduction.
- **The corpus is multilingual (~27% non-English** in the sampled window). An English-centric single-classifier agent would silently miss a large slice of traffic; language must be a first-class routing attribute.
- English traffic decomposes into a **12-intent taxonomy** (delivery delay, delivered-not-received, delivered-wrong-location, order-status, refund, cancellation, charge issue, return/replacement, device/app, account access, account info update, service-complaint/escalation).
- Intents **overlap naturally** (return↔refund, delay↔status) → Phase 3 labels should allow co-occurrence.
- Many messages (`thanks`, `still waiting`, `what?`) are **meaningless without context**; classify message+context, and treat acknowledgements as a separate messaging layer, not a domain intent.
- Brand replies are heavily **handoff-driven** (move to DM, provide help URL), so a later phase should reproduce *safe handoff* actions, not expose-resolution in a public reply.

**Out of scope (Phase 3+, deliberately not built here):** classifier, RAG, agent, golden eval split. This notebook only *discovers* the intent set and the data facts that constrain them.